In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('data/uy_ijarasi.csv')
print(data.head())

   maydon_m2  xonalar_soni  markazgacha_masofa_km  ijara_narxi_usd
0        106             3                   15.0              470
1         38             2                   17.9               80
2         36             1                    1.1              270
3        102             3                    4.4              609
4        108             4                    8.7              579


In [9]:
print("=== Dastlabki 10 qator ===")
print(data.head(10).to_string())
print("\n=== Ustun turlari ===")
print(data.dtypes)
print("\nTozalashdan oldin:", data.shape)
print("\n=== Bo'sh qiymatlar ===")
print(data.isna().sum())
print("\nTakroriy qatorlar soni:", data.duplicated().sum())
clean = data.dropna().drop_duplicates().reset_index(drop=True)
print("Tozalashdan keyin:", clean.shape)
clean.to_csv("uy_ijarasi_clean.csv", index=False)
stats = clean.agg(["mean", "median", "min", "max"]).round(2)
stats.index = ["o'rtacha", "mediana", "minimum", "maksimum"]
print("\n=== Statistika ===")
print(stats.to_string())

=== Dastlabki 10 qator ===
   maydon_m2  xonalar_soni  markazgacha_masofa_km  ijara_narxi_usd
0        106             3                   15.0              470
1         38             2                   17.9               80
2         36             1                    1.1              270
3        102             3                    4.4              609
4        108             4                    8.7              579
5        136             4                   16.2              640
6        114             4                    5.9              677
7         38             1                    2.4              288
8        133             4                    5.7              792
9         40             2                    7.9              217

=== Ustun turlari ===
maydon_m2                  int64
xonalar_soni               int64
markazgacha_masofa_km    float64
ijara_narxi_usd            int64
dtype: object

Tozalashdan oldin: (500, 4)

=== Bo'sh qiymatlar ===
maydon_m2    

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("uy_ijarasi_clean.csv")
A, R, D, P = "maydon_m2", "xonalar_soni", "markazgacha_masofa_km", "ijara_narxi_usd"
df["masofa_guruh"] = pd.cut(df[D], bins=[0, 5, 12, 20],
                           labels=["Yaqin (0-5 km)", "O'rta (5-12 km)", "Uzoq (12-20 km)"])
df["masofa_bin"] = pd.cut(df[D], bins=10)


def line(ax):
    g = df.groupby("masofa_bin", observed=True).agg(d=(D, "mean"), p=(P, "mean"))
    ax.plot(g["d"], g["p"], marker="o", color="#2a6f97")
    ax.set_title("1. Markazgacha masofa bo'yicha o'rtacha narx")
    ax.set_xlabel("Markazgacha masofa (km)")
    ax.set_ylabel("O'rtacha ijara narxi ($)")


def bar(ax):
    g = df.groupby(R)[P].mean()
    ax.bar(g.index.astype(str), g.values, color="#52b788")
    ax.set_title("2. Xonalar soni bo'yicha o'rtacha narx")
    ax.set_xlabel("Xonalar soni")
    ax.set_ylabel("O'rtacha ijara narxi ($)")


def hist(ax):
    sns.histplot(df[P], bins=25, kde=True, color="#e76f51", ax=ax)
    ax.set_title("3. Ijara narxi taqsimoti")
    ax.set_xlabel("Ijara narxi ($)")
    ax.set_ylabel("Uylar soni")


def scatter(ax):
    sc = ax.scatter(df[A], df[P], c=df[D], cmap="viridis", s=20, alpha=0.8)
    plt.colorbar(sc, ax=ax, label="Markazgacha masofa (km)")
    ax.set_title("4. Maydon va narx (rang: masofa)")
    ax.set_xlabel("Maydon (m²)")
    ax.set_ylabel("Ijara narxi ($)")


def box(ax):
    sns.boxplot(data=df, x=R, y=P, color="#a8dadc", ax=ax)
    ax.set_title("5. Xonalar soni bo'yicha narx tarqalishi")
    ax.set_xlabel("Xonalar soni")
    ax.set_ylabel("Ijara narxi ($)")


def pie(ax):
    c = df[R].value_counts().sort_index()
    ax.pie(c.values, labels=[f"{i} xona" for i in c.index], autopct="%1.1f%%",
           startangle=90, colors=sns.color_palette("pastel"))
    ax.set_title("6. Xonalar soni bo'yicha uylar ulushi")


def heatmap(ax):
    corr = df[[A, R, D, P]].corr()
    corr.columns = corr.index = ["Maydon", "Xonalar", "Masofa", "Narx"]
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title("7. Korrelyatsiya xaritasi")


def violin(ax):
    sns.violinplot(data=df, x="masofa_guruh", y=P, color="#cdb4db", inner="quartile", ax=ax)
    ax.set_title("8. Masofa guruhlari bo'yicha narx zichligi")
    ax.set_xlabel("Markazgacha masofa guruhi")
    ax.set_ylabel("Ijara narxi ($)")


charts = [("1_line_plot", line), ("2_bar_chart", bar), ("3_histogram", hist),
          ("4_scatter_plot", scatter), ("5_box_plot", box), ("6_pie_chart", pie),
          ("7_correlation_heatmap", heatmap), ("8_violin_plot", violin)]

for name, fn in charts:
    fig, ax = plt.subplots(figsize=(7, 5))
    fn(ax)
    fig.tight_layout()
    fig.savefig(f"{name}.png", dpi=150)
    plt.close(fig)

fig, axes = plt.subplots(4, 2, figsize=(14, 22))
for ax, (_, fn) in zip(axes.ravel(), charts):
    fn(ax)
fig.tight_layout()
fig.savefig("barcha_grafiklar.png", dpi=110)

print(df[[A, R, D, P]].corr().round(2)[P])
print(df.groupby(R)[P].mean().round(0))
print(df.groupby("masofa_guruh", observed=True)[P].agg(["mean", "median"]).round(0))
print(df[R].value_counts(normalize=True).sort_index().round(3))

maydon_m2                0.94
xonalar_soni             0.89
markazgacha_masofa_km   -0.38
ijara_narxi_usd          1.00
Name: ijara_narxi_usd, dtype: float64
xonalar_soni
1    150.0
2    285.0
3    458.0
4    634.0
5    768.0
Name: ijara_narxi_usd, dtype: float64
                  mean  median
masofa_guruh                  
Yaqin (0-5 km)   540.0   512.0
O'rta (5-12 km)  493.0   493.0
Uzoq (12-20 km)  362.0   356.0
xonalar_soni
1    0.144
2    0.254
3    0.246
4    0.232
5    0.124
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("uy_ijarasi_clean.csv")

X = df[["maydon_m2"]]
y = df["ijara_narxi_usd"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Train: {len(X_train)} qator, Test: {len(X_test)} qator")
print(f"Tenglama: narx = {model.intercept_:.2f} + {model.coef_[0]:.2f} * maydon")
print(f"MAE: {mae:.2f} $")
print(f"R2:  {r2:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax = axes[0]
ax.scatter(X_train["maydon_m2"], y_train, s=15, alpha=0.35, color="gray", label="Train")
ax.scatter(X_test["maydon_m2"], y_test, s=25, color="#2a6f97", label="Test (haqiqiy)")
xs = pd.DataFrame({"maydon_m2": np.linspace(X["maydon_m2"].min(), X["maydon_m2"].max(), 100)})
ax.plot(xs["maydon_m2"], model.predict(xs), color="#e63946", linewidth=2.5,
        label="Regressiya chizig'i")
ax.set_title("Maydon bo'yicha narx regressiyasi")
ax.set_xlabel("Maydon (m²)")
ax.set_ylabel("Ijara narxi ($)")
ax.legend()
ax = axes[1]
ax.scatter(y_test, y_pred, s=30, color="#52b788", alpha=0.85)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
ax.plot([lo, hi], [lo, hi], "--", color="#e63946", label="Ideal (y = x)")
ax.set_title(f"Haqiqiy va bashorat qilingan narx\nMAE = {mae:.1f} $, R² = {r2:.3f}")
ax.set_xlabel("Haqiqiy narx ($)")
ax.set_ylabel("Bashorat qilingan narx ($)")
ax.legend()
fig.tight_layout()
fig.savefig("regressiya_natija.png", dpi=150)
print("Grafik saqlandi: regressiya_natija.png")

Train: 400 qator, Test: 100 qator
Tenglama: narx = -81.34 + 6.01 * maydon
MAE: 71.42 $
R2:  0.8494
Grafik saqlandi: regressiya_natija.png


In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("uy_ijarasi_clean.csv")

y = df["ijara_narxi_usd"]

features = ["maydon_m2", "xonalar_soni", "markazgacha_masofa_km"]
X = df[features]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model2 = LinearRegression().fit(X_train, y_train)
pred2 = model2.predict(X_test)
mae2 = mean_absolute_error(y_test, pred2)
r2_2 = r2_score(y_test, pred2)

model1 = LinearRegression().fit(X_train[["maydon_m2"]], y_train)
pred1 = model1.predict(X_test[["maydon_m2"]])
mae1 = mean_absolute_error(y_test, pred1)
r2_1 = r2_score(y_test, pred1)

print(f"Train: {len(X_train)} qator, Test: {len(X_test)} qator")
print("\nKo'p o'zgaruvchili tenglama:")
print(f"narx = {model2.intercept_:.2f}", end="")
for name, c in zip(features, model2.coef_):
    print(f" {c:+.2f}*{name}", end="")
print()
print(f"\n1-model (faqat maydon):   MAE = {mae1:.2f} $,  R2 = {r2_1:.4f}")
print(f"2-model (3 ta o'zgaruvchi): MAE = {mae2:.2f} $,  R2 = {r2_2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
lo = min(y_test.min(), pred1.min(), pred2.min())
hi = max(y_test.max(), pred1.max(), pred2.max())
for ax, pred, mae, r2, title, color in [
    (axes[0], pred1, mae1, r2_1, "1-model: faqat maydon", "#2a6f97"),
    (axes[1], pred2, mae2, r2_2, "2-model: maydon + xonalar + masofa", "#52b788"),
]:
    ax.scatter(y_test, pred, s=30, color=color, alpha=0.85)
    ax.plot([lo, hi], [lo, hi], "--", color="#e63946", label="Ideal (y = x)")
    ax.set_title(f"{title}\nMAE = {mae:.1f} $, R² = {r2:.3f}")
    ax.set_xlabel("Haqiqiy narx ($)")
    ax.legend()
axes[0].set_ylabel("Bashorat qilingan narx ($)")
fig.tight_layout()
fig.savefig("regressiya_multi_natija.png", dpi=150)
print("Grafik saqlandi: regressiya_multi_natija.png")

Train: 400 qator, Test: 100 qator

Ko'p o'zgaruvchili tenglama:
narx = 69.18 +4.89*maydon_m2 +29.70*xonalar_soni -13.58*markazgacha_masofa_km

1-model (faqat maydon):   MAE = 71.42 $,  R2 = 0.8494
2-model (3 ta o'zgaruvchi): MAE = 20.88 $,  R2 = 0.9838
Grafik saqlandi: regressiya_multi_natija.png


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("uy_ijarasi_clean.csv")
features = ["maydon_m2", "xonalar_soni", "markazgacha_masofa_km"]
X, y = df[features], df["ijara_narxi_usd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

m1 = LinearRegression().fit(X_train[["maydon_m2"]], y_train)
m2 = LinearRegression().fit(X_train, y_train)

p1, p2 = m1.predict(X_test[["maydon_m2"]]), m2.predict(X_test)
res = pd.DataFrame({
    "Ko'rsatkich": ["O'zgaruvchilar", "MAE ($)", "R²", "Xato (MAE) nisbati"],
    "1-model (bir o'zgaruvchili)": ["maydon", f"{mean_absolute_error(y_test, p1):.2f}",
                                    f"{r2_score(y_test, p1):.4f}", "1.00x"],
    "2-model (ko'p o'zgaruvchili)": ["maydon, xonalar, masofa",
                                     f"{mean_absolute_error(y_test, p2):.2f}",
                                     f"{r2_score(y_test, p2):.4f}",
                                     f"{mean_absolute_error(y_test, p2) / mean_absolute_error(y_test, p1):.2f}x"],
})
print(res.to_string(index=False))

new = pd.DataFrame({
    "maydon_m2": [45, 70, 95, 130],
    "xonalar_soni": [1, 2, 3, 4],
    "markazgacha_masofa_km": [3.0, 8.0, 12.0, 6.0],
})
new["1-model ($)"] = m1.predict(new[["maydon_m2"]]).round(0)
new["2-model ($)"] = m2.predict(new[features]).round(0)
new["Formula ($)"] = (60 + 5 * new["maydon_m2"] + 30 * new["xonalar_soni"]
                      - 14 * new["markazgacha_masofa_km"]).round(0)
print()
print(new.to_string(index=False))

       Ko'rsatkich 1-model (bir o'zgaruvchili) 2-model (ko'p o'zgaruvchili)
    O'zgaruvchilar                      maydon      maydon, xonalar, masofa
           MAE ($)                       71.42                        20.88
                R²                      0.8494                       0.9838
Xato (MAE) nisbati                       1.00x                        0.29x

 maydon_m2  xonalar_soni  markazgacha_masofa_km  1-model ($)  2-model ($)  Formula ($)
        45             1                    3.0        189.0        278.0        273.0
        70             2                    8.0        339.0        362.0        358.0
        95             3                   12.0        489.0        460.0        457.0
       130             4                    6.0        700.0        742.0        746.0
